# Unit 12B — Flow matching: objective, Euler sampler and the policy hook

**Your result:** the training objective and the sampler of SmolVLA's action expert, written by you and running inside the real model. Your loss equals the pinned LeRobot loss for the same noise and times; your Euler loop equals the pinned integrator; and a real policy wrapper uses your sampler to produce the actions it hands to the simulator.

Four short exercises (10–20 minutes each) and a larger experiment (30–60 minutes). These times are estimates, not measurements. Full answers are in [the reference notebook](12b-flow-reference.ipynb).

**Sources.** [SmolVLA method description](https://huggingface.co/blog/smolvla#method) (flow-matching action expert; the VLM is frozen); the pinned code in `lerobot/policies/smolvla/modeling_smolvla.py` (`VLAFlowMatching.forward`, `sample_actions`, `denoise_step`) and `lerobot/policies/common/flow_matching.py` (`euler_integrate`), revision `e0d50211ef236143ae867228662b7dfaba554f02`. Flow matching: [Lipman et al. 2023](https://arxiv.org/abs/2210.02747).

**Do not mix up two things.** The SmolVLA *action expert* is a small language-model-style stack that attends to the VLM's cached keys and values. It is **not** the ACT Transformer of Unit 8: do not assume they are the same computation. And flow matching here is **not** DDPM: there is no noise schedule `ᾱ`, no clipping and no random step noise. The sampler is deterministic given the starting noise.

**Where this unit sits.** The six Unit 12A functions are available as package versions. The *raw velocity adapter* (`SmolVLAAdapter.velocity`, the pinned model's forward pass up to the velocity) is supplied and tested against the pinned `VLAFlowMatching.forward`; you write the objective around it.

In [ ]:
from __future__ import annotations


from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from collections.abc import Callable
    from collections.abc import Mapping
    from collections.abc import Sequence
    from torch import Tensor
    from vla_lab.teaching.interfaces import VLAInputs
    from vla_lab.teaching.records import TrialConfig
    from vla_lab.teaching.records import TrialResult


# Supplied: workspace, offline caches, imports
from pathlib import Path
import os
import sys


def find_root() -> Path:
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the repository tree.")


WORKSPACE = find_root()
if str(WORKSPACE / "src") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "src"))

# Set cache paths and offline mode before any data or model library is imported.
os.environ["HF_HUB_CACHE"] = str(WORKSPACE / ".local-data/hf/hub")
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / ".local-data/hf/datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ.setdefault("TORCH_HOME", str(WORKSPACE / ".local-data/torch"))
CONFIGS, DATA = WORKSPACE / "configs", WORKSPACE / ".local-data"
import json
import shutil
import tempfile
import time
from pathlib import Path

import torch
from torch import nn
import matplotlib.pyplot as plt
import yaml
from transformers import AutoTokenizer

import vla_lab.teaching.smolvla as ref  # package reference: used by checks only
from vla_lab.teaching.smolvla import (  # Unit 12A functions, as package versions
    CallbackSmolVLAPolicy, SmolVLAAdapter, prepare_vla_inputs, select_trainable_parameters,
    train_mode_with_frozen_vlm, pad_to_width, load_prepared_smolvla,
)
from vla_lab.teaching.act_model import load_act_samples, open_libero_dataset
from vla_lab.teaching.training import masked_action_l1, optimizer_update, train_loop
from lerobot.configs import FeatureType, PolicyFeature
from lerobot.policies.common.flow_matching import euler_integrate
from lerobot.policies.smolvla import SmolVLAConfig, SmolVLAPolicy

torch.manual_seed(0)
print("PyTorch:", torch.__version__)
print("Workspace:", WORKSPACE)

In [ ]:
# Supplied: a helper that the checks use
def close(actual: torch.Tensor, expected: torch.Tensor, message: str, atol: float=1e-5) -> None:
    """Compare two tensors and say what failed."""
    assert actual.shape == expected.shape, f"{message}: shape {tuple(actual.shape)}, expected {tuple(expected.shape)}"
    worst = (actual.detach() - expected.detach()).abs().max().item()
    assert worst <= atol, f"{message}: largest difference {worst:.2e} is above {atol:.0e}"

## Background: the flow and its sign convention

Flow matching learns a **velocity field** that moves noise to actions along straight lines. SmolVLA uses time `t ∈ [0, 1]` with this convention (it is the pinned one; the opposite convention is common in other code, so check signs):

- `t = 0` is the **clean action chunk**; `t = 1` is **pure noise**.
- Training pair: `x_t = t · noise + (1 − t) · actions` and target velocity `u_t = noise − actions`. (Because `d x_t / d t = noise − actions`.)
- The expert predicts `v(x_t, t, observation)`; the loss is the mean squared error `‖v − u_t‖²` over real action steps and real action columns.
- **Sampling** integrates from `t = 1` (noise) to `t = 0` with forward Euler: `dt = −1 / n`, `x ← x + dt · v(x, t)`, `t = 1, 1 − 1/n, …`.
- Actions are zero-padded to `max_action_dim = 32` columns inside the model. Only the first `action_dim = 7` columns and the non-padded time steps count in the loss.
- Training times come from a prior that favors noisy times: `t = 0.001 + 0.999 · Beta(1.5, 1)`.

**Real data used in this unit.** Real LIBERO `bowl_plate` frames, states and action chunks, read from local files (no download starts here). The recording rate is 10 Hz; nothing is resampled to the simulator's 20 Hz controller. If files are missing, the cell tells you the setup command.

In [ ]:
# Supplied: a small real SmolVLA (the pinned classes, a tiny VLM, random weights) and its tokenizer
VLM_DIR = DATA / "models/smolvlm2-500m-video-instruct"     # local tokenizer files and config; the 500M weights are NOT read here
IMAGES = ("observation.images.image", "observation.images.image2")
PROMPT = "put the bowl on the plate"


def make_tiny_smolvla(seed: int=1) -> SmolVLAPolicy:
    """A small *real* pinned SmolVLAPolicy: the same classes and code path as the 450M model,
    with a tiny VLM (width 64, 2 layers, 64x64 images). Random weights; it checks mechanics only."""
    tiny = Path(tempfile.mkdtemp(prefix="tiny_smolvlm_"))
    for path in VLM_DIR.iterdir():
        if path.is_file() and path.name != "model.safetensors":
            shutil.copy(path, tiny / path.name)
    vlm = json.loads((VLM_DIR / "config.json").read_text())
    vlm["text_config"].update(hidden_size=64, intermediate_size=128, num_attention_heads=4,
                              num_key_value_heads=2, head_dim=16, num_hidden_layers=2)
    vlm["vision_config"].update(hidden_size=32, num_attention_heads=2, num_hidden_layers=1, intermediate_size=64,
                                image_size=64, patch_size=16, max_image_size={"longest_edge": 64}, size={"longest_edge": 64})
    vlm["scale_factor"] = 2
    (tiny / "config.json").write_text(json.dumps(vlm))
    config = SmolVLAConfig(
        device="cpu", push_to_hub=False, vlm_model_name=str(tiny), load_vlm_weights=False,
        input_features={
            IMAGES[0]: PolicyFeature(FeatureType.VISUAL, (3, 64, 64)),
            IMAGES[1]: PolicyFeature(FeatureType.VISUAL, (3, 64, 64)),
            "observation.state": PolicyFeature(FeatureType.STATE, (8,)),
        },
        output_features={"action": PolicyFeature(FeatureType.ACTION, (7,))},
        chunk_size=10, n_action_steps=5, num_vlm_layers=2, resize_imgs_with_padding=(64, 64),
        max_state_dim=16, max_action_dim=16, prefix_length=0, num_steps=4, expert_width_multiplier=0.5,
        tokenizer_max_length=24,
    )
    torch.manual_seed(seed)
    return SmolVLAPolicy(config)


TOKENIZER = AutoTokenizer.from_pretrained(str(VLM_DIR))
POLICY = make_tiny_smolvla()
print("tiny SmolVLA parameters:", sum(p.numel() for p in POLICY.parameters()))

In [ ]:
# Supplied: a small batch for algorithm checks (random numbers) in the shapes the model reads
def algorithm_check_batch(batch: int=2, chunk: int=10, seed: int=0) -> dict[str, torch.Tensor]:
    g = torch.Generator().manual_seed(seed)
    tokens, mask = ref.tokenize_prompts(TOKENIZER, ["put the bowl on the plate", "open it"], 24)
    pad = torch.zeros(batch, chunk, dtype=torch.bool)
    pad[1, 7:] = True
    return {
        IMAGES[0]: torch.rand(batch, 3, 64, 64, generator=g), IMAGES[1]: torch.rand(batch, 3, 64, 64, generator=g),
        "observation.state": torch.randn(batch, 8, generator=g),
        "observation.language.tokens": tokens, "observation.language.attention_mask": mask,
        "action": torch.randn(batch, chunk, 7, generator=g), "action_is_pad": pad,
    }

CHECK_BATCH = algorithm_check_batch()

## Exercise 12B.1 — The flow training pair (about 10 min)

**Purpose.** Make the noisy input and the velocity target for flow matching. Getting the direction or the sign wrong trains a model that moves actions the wrong way.

**Contract (shapes, units, masks).**

- `flow_training_pair(actions, noise, time)`: `actions`, `noise`: `(B, T, D)`; `time`: `(B,)` `float32` in `[0, 1]`.
- Return `(x_t, u_t)`, both `(B, T, D)`:
  `x_t = t * noise + (1 - t) * actions` and `u_t = noise - actions`, with each row using its own `t`.
- `t = 0` gives the clean actions; `t = 1` gives the noise.

**Method and source.** Linear interpolation path of [flow matching](https://arxiv.org/abs/2210.02747). Pinned: `x_t = time_expanded * noise + (1 - time_expanded) * actions` and `u_t = noise - actions` in `VLAFlowMatching.forward` (the opposite of the usual diffusion-literature direction).

**Pointers.** Reshape `time` to `(B, 1, 1)` so that it broadcasts. The target does not depend on time.

<details><summary>Detailed hint</summary>

```python
t = time.reshape(-1, 1, 1)
return t * noise + (1.0 - t) * actions, noise - actions
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 12B.1
from __future__ import annotations

def flow_training_pair(actions: torch.Tensor, noise: torch.Tensor, time: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Noisy actions and velocity target for flow matching.

    ``actions``, ``noise``: ``(B, T, D)``; ``time``: ``(B,)`` in ``[0, 1]``. Returns
    ``x_t = t * noise + (1 - t) * actions`` and ``u_t = noise - actions``, both ``(B, T, D)``.
    """
    t = time.reshape(-1, 1, 1)
    return t * noise + (1.0 - t) * actions, noise - actions


In [ ]:
# Check: Exercise 12B.1
actions, noise = torch.randn(3, 6, 4), torch.randn(3, 6, 4)
x0, u = flow_training_pair(actions, noise, torch.zeros(3))
assert x0.shape == actions.shape and u.shape == actions.shape, "Both outputs are (B, T, D)."
assert torch.equal(x0, actions), "At t = 0 the sample is the clean actions."
close(u, noise - actions, "the target is noise - actions (not actions - noise)")
x1, _ = flow_training_pair(actions, noise, torch.ones(3))
close(x1, noise, "At t = 1 the sample is the noise")
x, _ = flow_training_pair(actions, noise, torch.tensor([0.25, 0.5, 0.75]))
close(x[0], 0.25 * noise[0] + 0.75 * actions[0], "row 0 uses its own t = 0.25")
close(x[2], 0.75 * noise[2] + 0.25 * actions[2], "row 2 uses its own t = 0.75")
# the velocity is the derivative of the path: moving along -u from t=1 reaches the actions
t = torch.tensor([0.4, 0.4, 0.4])
xt, ut = flow_training_pair(actions, noise, t)
close(xt - 0.4 * ut, actions, "x_t - t * u_t is the clean action: the path is straight", atol=1e-6)
print("Exercise 12B.1 passed")

**Use your function.** Draw the straight path from real actions to noise for one action channel.

In [ ]:
# Use your function: Exercise 12B.1
real = load_act_samples(CONFIGS, DATA, "bowl_plate", "validation", count=4, chunk_size=10, image_size=64)
actions = real["action"][:1, :, :7]
noise = torch.randn(actions.shape, generator=torch.Generator().manual_seed(0))
fig, axis = plt.subplots(figsize=(6.5, 3.2))
for t in (0.0, 0.25, 0.5, 0.75, 1.0):
    x_t, u_t = flow_training_pair(actions, noise, torch.tensor([t]))
    axis.plot(x_t[0, :, 0], label=f"t={t}")
axis.set_xlabel("action step"); axis.set_ylabel("channel 0"); axis.legend(ncol=3, fontsize=8); plt.show()

## Exercise 12B.2 — Masked velocity loss (about 10–15 min)

**Purpose.** The model works with 32 action columns and a fixed chunk length, but only 7 columns and the non-padded steps are real. The loss must ignore the rest.

**Contract (shapes, units, masks).**

- `masked_velocity_loss(prediction, target, is_pad, action_dim)`: `prediction`, `target`: `(B, T, D_max)`; `is_pad`: `(B, T)` bool, `True` = padded step; `action_dim <= D_max`.
- Squared error `(prediction - target)**2` on the **first `action_dim` columns only**, zero on padded steps.
- Return `sum(error * valid) / max(valid_steps * action_dim, 1)` where `valid_steps` is the number of non-padded `(row, step)` pairs in the whole batch.
- Changing `prediction` or `target` in padded steps or in columns `>= action_dim` must change neither the loss nor the gradient. An all-padding batch returns `0`.

**Method and source.** Pinned `SmolVLAPolicy.forward`: `losses = losses[:, :, :action_dim]`, multiply by `~action_is_pad`, then `losses.sum() / (num_valid * losses.shape[-1]).clamp_min(1)`.

**Pointers.** Slice the columns first. Then use a float mask of shape `(B, T, 1)`. The denominator counts valid *scalar* values.

<details><summary>Detailed hint</summary>

```python
squared = (prediction[..., :action_dim] - target[..., :action_dim]).square()
valid = (~is_pad.to(torch.bool)).unsqueeze(-1).to(squared.dtype)
return (squared * valid).sum() / (valid.sum() * action_dim).clamp_min(1)
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 12B.2
from __future__ import annotations

def masked_velocity_loss(
    prediction: torch.Tensor, target: torch.Tensor, is_pad: torch.Tensor, action_dim: int
) -> torch.Tensor:
    """Mean squared velocity error over real time steps and real action columns.

    ``prediction`` and ``target``: ``(B, T, D_max)``; ``is_pad``: ``(B, T)``, ``True`` for padding;
    ``action_dim <= D_max`` is the number of real action columns (the rest are zero padding).
    ``sum(error ** 2 over [:, :, :action_dim] * ~is_pad) / max(valid_steps * action_dim, 1)``.
    """
    squared = (prediction[..., :action_dim] - target[..., :action_dim]).square()
    valid = (~is_pad.to(torch.bool)).unsqueeze(-1).to(squared.dtype)
    return (squared * valid).sum() / (valid.sum() * action_dim).clamp_min(1)


In [ ]:
# Check: Exercise 12B.2
pred = torch.zeros(2, 3, 5, requires_grad=True)
target = torch.ones(2, 3, 5)
pad = torch.tensor([[False, False, True], [False, True, True]])
loss = masked_velocity_loss(pred, target, pad, 3)
assert loss.shape == () and torch.isfinite(loss), "Return one finite scalar."
close(loss, torch.tensor(1.0), "three real steps, three real columns, error 1")
loss.backward()
assert torch.count_nonzero(pred.grad[pad]) == 0, "Padded steps need zero gradient."
assert torch.count_nonzero(pred.grad[..., 3:]) == 0, "Padded action columns need zero gradient."
other = pred.detach().clone()
other[..., 3:] = 50.0
other[pad] = 50.0
close(masked_velocity_loss(other, target, pad, 3), loss.detach(), "changing padded steps or columns must not change the loss")
assert masked_velocity_loss(pred.detach(), target, torch.ones_like(pad), 3) == 0, "An all-padding batch gives exactly 0."
wide = masked_velocity_loss(torch.zeros(1, 2, 7), torch.ones(1, 2, 7), torch.zeros(1, 2, dtype=torch.bool), 7)
close(wide, torch.tensor(1.0), "all columns real")
print("Exercise 12B.2 passed")

**Use your function.** On a real batch, build a prediction that is exact on real steps and columns and wrong only where they are padded: the loss must be exactly zero.

In [ ]:
# Use your function: Exercise 12B.2
real = load_act_samples(CONFIGS, DATA, "bowl_plate", "validation", count=16, chunk_size=10, image_size=64)
target = pad_to_width(real["action"], 32) * 0.0 + torch.randn(16, 10, 32, generator=torch.Generator().manual_seed(1))
prediction = target.clone()
prediction[real["action_is_pad"]] = 0.0
prediction[..., 7:] = 9.0
print("padded steps in this batch:", int(real["action_is_pad"].sum()), "of", real["action_is_pad"].numel())
print("loss:", float(masked_velocity_loss(prediction, target, real["action_is_pad"], 7)), "(0 means only real steps and columns count)")

## Exercise 12B.3 — The loss callback with explicit noise and time (about 20 min)

**Purpose.** Connect your pair and your loss to the real model. The training loop calls this function; passing the noise and the time explicitly makes every update repeatable and lets you compare with the pinned loss.

**Contract (shapes, units, masks).**

- `flow_loss_fn(model, batch, *, noise=None, time=None) -> (scalar_loss, metrics)`. The loss keeps its autograd graph; `metrics` is a dict of scalar tensors.
- `model` is a `SmolVLAAdapter`. `inputs = model.prepare_inputs(batch)` gives a dict with `inputs["actions"]` already zero-padded to `(B, T, D_max)`.
- `noise`: `(B, T, D_max)` standard normal noise. If `None`, draw `torch.randn_like(actions)`. If given with another shape, raise `ValueError`.
- `time`: `(B,)` float in `(0, 1)`. If `None`, use `model.sample_time(B, device)` (the pinned Beta prior).
- `x_t, u_t = flow_training_pair(actions, noise, time)`; `v = model.velocity(inputs, x_t, time)` (shape `(B, T, D_max)`); `loss = masked_velocity_loss(v, u_t, batch["action_is_pad"], model.action_dim)`.
- Return `loss, {"flow_mse": loss}`.

**Method and source.** Pinned `SmolVLAPolicy.forward` / `VLAFlowMatching.forward`: noise and time may be passed in; otherwise noise is standard normal and time follows the Beta prior; the loss is masked as in Exercise 12B.2. The velocity adapter (`SmolVLAAdapter.velocity`) re-implements the pinned forward pass up to the velocity so that this loss sits on top of the *real* expert.

**Pointers.** Get the inputs, then fill in the missing noise and time, then the three calls. Do not detach anything.

<details><summary>Detailed hint</summary>

```python
inputs = model.prepare_inputs(batch)
actions = inputs["actions"]
if noise is None:
    noise = torch.randn_like(actions)
if noise.shape != actions.shape:
    raise ValueError(f"noise must have the padded action shape {tuple(actions.shape)}")
if time is None:
    time = model.sample_time(actions.shape[0], actions.device)
x_t, target = flow_training_pair(actions, noise, time)
velocity = model.velocity(inputs, x_t, time)
loss = masked_velocity_loss(velocity, target, batch["action_is_pad"], model.action_dim)
return loss, {"flow_mse": loss}
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 12B.3
from __future__ import annotations

def flow_loss_fn(
    model: SmolVLAAdapter,
    batch: Mapping[str, torch.Tensor],
    *,
    noise: torch.Tensor | None = None,
    time: torch.Tensor | None = None,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    """Loss callback ``(scalar, metrics)`` that keeps the gradient path through the real model.

    ``model`` is a ``SmolVLAAdapter``. Build the inputs with ``model.prepare_inputs(batch)``
    (actions come back zero padded to ``max_action_dim``). Use ``noise`` ``(B, T, max_action_dim)`` and
    ``time`` ``(B,)`` when given; otherwise draw standard normal noise and ``model.sample_time``.
    Make ``(x_t, u_t)`` with ``flow_training_pair``, get ``v = model.velocity(inputs, x_t, time)`` and return
    ``masked_velocity_loss(v, u_t, batch["action_is_pad"], model.action_dim)``.
    """
    inputs = model.prepare_inputs(batch)
    actions = inputs["actions"]
    if noise is None:
        noise = torch.randn_like(actions)
    if noise.shape != actions.shape:
        raise ValueError(f"noise must have the padded action shape {tuple(actions.shape)}")
    if time is None:
        time = model.sample_time(actions.shape[0], actions.device)
    x_t, target = flow_training_pair(actions, noise, time)
    velocity = model.velocity(inputs, x_t, time)
    loss = masked_velocity_loss(velocity, target, batch["action_is_pad"], model.action_dim)
    return loss, {"flow_mse": loss}


In [ ]:
# Check: Exercise 12B.3
policy = make_tiny_smolvla(seed=1)
adapter = SmolVLAAdapter(policy)
noise, time = torch.randn(2, 10, 16), torch.tensor([0.3, 0.6])
expected, _ = policy(dict(CHECK_BATCH), noise=noise, time=time)           # the pinned loss
loss, metrics = flow_loss_fn(adapter, CHECK_BATCH, noise=noise, time=time)
assert loss.shape == () and "flow_mse" in metrics, "Return (scalar loss, metrics with flow_mse)."
close(loss, expected, "your loss differs from the pinned SmolVLA loss for the same noise and time", atol=1e-6)
again, _ = flow_loss_fn(adapter, CHECK_BATCH, noise=noise, time=time)
close(again, loss.detach(), "explicit noise and time must make the loss repeatable", atol=1e-7)
loss.backward()
assert any(p.grad is not None and p.grad.abs().sum() > 0 for p in policy.model.vlm_with_expert.lm_expert.parameters()), "The loss must reach the action expert (keep the graph)."
assert all(p.grad is None for p in policy.model.vlm_with_expert.vlm.parameters()), "The frozen VLM must get no gradient."
try:
    flow_loss_fn(adapter, CHECK_BATCH, noise=torch.randn(2, 10, 7), time=time)
except ValueError:
    pass
else:
    raise AssertionError("Noise with the wrong shape must raise ValueError.")
drawn, _ = flow_loss_fn(adapter, CHECK_BATCH)                             # no noise, no time: drawn inside
assert torch.isfinite(drawn), "Draw noise and time when they are not given."
all_pad = dict(CHECK_BATCH, action_is_pad=torch.ones(2, 10, dtype=torch.bool))
assert flow_loss_fn(adapter, all_pad, noise=noise, time=time)[0] == 0, "An all-padding batch has zero loss."
print("Exercise 12B.3 passed")

**Use your function.** Update the expert with the training unit's `optimizer_update`, which calls your callback. The frozen VLM must stay exactly where it is, and the loss on this batch should fall.

In [ ]:
# Use your function: Exercise 12B.3
policy = make_tiny_smolvla(seed=1)
for parameter in policy.parameters():
    parameter.requires_grad_(True)
params = select_trainable_parameters(policy)                               # Unit 12A (package version)
adapter = SmolVLAAdapter(policy, train_mode_fn=train_mode_with_frozen_vlm)
optimizer = torch.optim.AdamW(params, lr=1e-3)
vlm_before = {n: p.detach().clone() for n, p in policy.model.vlm_with_expert.vlm.named_parameters()}
noise, time = torch.randn(2, 10, 16, generator=torch.Generator().manual_seed(0)), torch.tensor([0.4, 0.7])
history = [optimizer_update(adapter, CHECK_BATCH, optimizer, flow_loss_fn, noise=noise, time=time)["loss"] for _ in range(20)]
print("loss, first and last update:", round(history[0], 4), round(history[-1], 4))
print("VLM unchanged:", all(torch.equal(vlm_before[n], p) for n, p in policy.model.vlm_with_expert.vlm.named_parameters()))

## Exercise 12B.4 — The Euler sampler from noise to actions (about 15 min)

**Purpose.** Turn the learned velocity into actions by integrating from `t = 1` to `t = 0`. Time runs *backwards*; the step is negative.

**Contract (shapes, units, masks).**

- `euler_sample(velocity_fn, noise, num_steps)`: `noise (B, T, D)`; `velocity_fn(x_t, t) -> v` where `t` is a **`float32` tensor `(B,)`** filled with the current time and `v` has the shape of `x_t`.
- `dt = -1.0 / num_steps`. Start at `x = noise`. For `step = 0 … num_steps - 1`: `t = 1.0 + step * dt`; `x = x + dt * velocity_fn(x, t)`. The velocity is therefore evaluated at `t = 1, 1 - 1/n, …, 1/n` (never at 0).
- Return `x` `(B, T, D)`. No randomness inside.

**Method and source.** Forward Euler for the ODE `dx/dt = v` from `t = 1` to `t = 0`; pinned `euler_integrate` in `flow_matching.py` (`dt = -1/num_steps`, `time = 1.0 + step * dt`). If `v = noise - actions` exactly, one Euler step already lands on the actions (straight path).

**Pointers.** A for-loop with one update line. Build the time tensor with `torch.full((batch,), t, dtype=torch.float32, device=x.device)`.

<details><summary>Detailed hint</summary>

```python
dt = -1.0 / num_steps
x = noise
for step in range(num_steps):
    t = torch.full((x.shape[0],), 1.0 + step * dt, dtype=torch.float32, device=x.device)
    x = x + dt * velocity_fn(x, t)
return x
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 12B.4
from __future__ import annotations

def euler_sample(
    velocity_fn: Callable[[torch.Tensor, torch.Tensor], torch.Tensor], noise: torch.Tensor, num_steps: int
) -> torch.Tensor:
    """Integrate the learned velocity from noise (``t = 1``) to actions (``t = 0``).

    ``velocity_fn(x_t, t)`` takes ``x_t (B, T, D)`` and ``t`` as a ``float32`` ``(B,)`` tensor and
    returns ``v`` of the same shape as ``x_t``. With ``dt = -1 / num_steps`` repeat
    ``num_steps`` times: ``t = 1 + step * dt``; ``x = x + dt * velocity_fn(x, t)``.
    The last velocity is evaluated at ``t = 1 / num_steps``, not at ``0``.
    """
    dt = -1.0 / num_steps
    x = noise
    for step in range(num_steps):
        t = torch.full((x.shape[0],), 1.0 + step * dt, dtype=torch.float32, device=x.device)
        x = x + dt * velocity_fn(x, t)
    return x


In [ ]:
# Check: Exercise 12B.4
noise, field = torch.randn(2, 6, 4), torch.randn(2, 6, 4)
seen = []

def velocity(x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    seen.append(t.clone())
    return field + 0.1 * x

mine = euler_sample(velocity, noise, 5)
assert [round(float(t[0]), 6) for t in seen] == [1.0, 0.8, 0.6, 0.4, 0.2], "Times must run 1, 0.8, ..., 0.2 (never 0)."
assert all(t.dtype == torch.float32 and t.shape == (2,) for t in seen), "t must be a float32 tensor of shape (B,)."
close(mine, euler_integrate(lambda x, t: field + 0.1 * x, noise, 5), "differs from the pinned euler_integrate", atol=1e-6)
actions = torch.randn(2, 6, 4)
for steps in (1, 3, 10):
    close(euler_sample(lambda x, t: noise - actions, noise, steps), actions, f"the straight-line field is exact in {steps} step(s)", atol=1e-5)
assert torch.equal(euler_sample(velocity, noise, 4), euler_sample(velocity, noise, 4)), "No randomness inside the sampler."
print("Exercise 12B.4 passed")

**Use your function.** Sample actions from the small real model with your sampler through the supplied policy hook, and compare with the pinned `sample_actions` for the same noise. The weights are random, so only the *agreement* matters.

In [ ]:
# Use your function: Exercise 12B.4
policy = make_tiny_smolvla(seed=1).eval()
adapter = SmolVLAAdapter(policy)
inputs = adapter.prepare_inputs(CHECK_BATCH)
noise = torch.randn(2, 10, 16, generator=torch.Generator().manual_seed(0))
mine = adapter.sample_chunk(inputs, euler_sample, noise, 4)                  # your sampler, the real prefix cache and expert
with torch.no_grad():
    pinned = policy.model.sample_actions(inputs["images"], inputs["img_masks"], inputs["lang_tokens"], inputs["lang_masks"],
                                         inputs["state"], noise=noise)[:, :, :7]
print("chunk shape:", tuple(mine.shape), "| largest difference from the pinned sampler:", float((mine - pinned).abs().max()))
wrapper = CallbackSmolVLAPolicy(policy, euler_sample, num_steps=4, seed=1)    # a policy object for the simulator loop
observation = {k: v for k, v in CHECK_BATCH.items() if k not in ("action", "action_is_pad")}
actions = [wrapper.select_action(observation) for _ in range(6)]
print("actions per call:", tuple(actions[0].shape), "| chunks sampled for 6 calls:", wrapper.chunks_made, "(n_action_steps = 5)")

## Experiment — time prior and Euler steps (30–60 min)

**Question.** SmolVLA trains with a Beta time prior that favors noisy times and samples with 10 Euler steps. With a small real SmolVLA trained by your loss, how do the training time prior and the number of Euler steps change the executed actions and the time per chunk?

**Setup (real data, small real model).** 128 training and 64 validation windows from the frozen `bowl_plate` episodes (64×64 frames, standardized state, 10 standardized actions, padding at episode ends, the instruction `put the bowl on the plate`). The expert and projections are trained for 300 updates with **your** `flow_loss_fn` (through `optimizer_update` with explicit noise and time) after your Unit 12A parameter selection; the sampler is **your** `euler_sample` through the adapter's prefix cache. The VLM has random weights: this is a mechanics experiment, not a robot policy.

**Fixed for every trial.** Weight seed `1`; batch/noise seed `1`; **300 updates**, batch 16, AdamW `lr=1e-3`; validation initial noise seed `11` (the same noise for every step count); first 5 actions of the chunk are the "executed" ones (`n_action_steps = 5`).

**Allowed changes (per trial).** `time_prior`: `beta` (pinned: `0.001 + 0.999 · Beta(1.5, 1)`) or `uniform` (`0.001 + 0.998 · U(0, 1)`) — this retrains the model. `steps`: Euler steps at inference, any of `1, 2, 5, 10, 20, 50`.

**Measures.** `l1_to_demo`: masked L1 between the executed actions and the demonstrated first 5 actions. `l1_to_50`: L1 between the executed actions and those of the 50-step sample of the same model and noise. `ms_batch1`: milliseconds to sample one chunk at batch 1 (noisy; compare within one session).

**Prediction first.** In the next cell: which `time_prior` will be better, and where will extra Euler steps stop helping?

**Keep failed trials.** Every trial is appended to `TRIALS`. No improvement is a valid result.

In [ ]:
# Larger training experiment: run only when you opt in.
RUN_EXPERIMENT = False

In [ ]:
if RUN_EXPERIMENT:
    PREDICTION = "TODO: replace this text with your prediction."
    print(PREDICTION)

In [ ]:
if RUN_EXPERIMENT:
    # Supplied: trial table helper
    import time
    import pandas as pd

    def show_trials(trials: Sequence[TrialResult], columns: Sequence[str]) -> pd.DataFrame:
        """Print every trial that was run, including failed ones, and return a table."""
        table = pd.DataFrame([{**t["config"], **{c: t.get(c) for c in columns}, "status": t["status"]} for t in trials])
        print(table.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
        return table

In [ ]:
if RUN_EXPERIMENT:
    # Supplied: real windows for the small model
    TRAIN = load_act_samples(CONFIGS, DATA, "bowl_plate", "train", count=128, chunk_size=10, image_size=64)
    VALID = load_act_samples(CONFIGS, DATA, "bowl_plate", "validation", count=64, chunk_size=10, image_size=64)
    print("train windows:", len(TRAIN["action"]), "| validation windows:", len(VALID["action"]))

In [ ]:
if RUN_EXPERIMENT:
    # Supplied: harness. Your loss callback and your sampler are in the real path.
    STEPS, BATCH, LR, SEED_WEIGHTS, SEED_BATCH, SEED_NOISE, N_ACT = 300, 16, 1e-3, 1, 1, 11, 5
    BASELINE = {"time_prior": "beta", "steps": 10}
    TRIALS, TRAINED, REFERENCE_CHUNKS = [], {}, {}


    def input_fn(adapter: SmolVLAAdapter, raw: Mapping[str, torch.Tensor]) -> VLAInputs:
        observation = {key: value for key, value in raw.items() if key.startswith("observation.")}
        inputs = prepare_vla_inputs(observation, PROMPT, TOKENIZER, adapter.policy.config)
        if "action" in raw:
            inputs["actions"] = pad_to_width(raw["action"], adapter.policy.config.max_action_dim)
        return inputs


    def train_model(time_prior: str) -> SmolVLAAdapter:
        """Train once per time prior (cached): your flow loss, explicit noise and time, seeded."""
        if time_prior in TRAINED:
            return TRAINED[time_prior]
        policy = make_tiny_smolvla(seed=SEED_WEIGHTS)
        for parameter in policy.parameters():
            parameter.requires_grad_(True)
        params = select_trainable_parameters(policy)
        adapter = SmolVLAAdapter(policy, train_mode_fn=train_mode_with_frozen_vlm, input_fn=input_fn)
        optimizer = torch.optim.AdamW(params, lr=LR)
        torch.manual_seed(SEED_BATCH)
        for _ in range(STEPS):
            pick = torch.randint(0, len(TRAIN["action"]), (BATCH,))
            batch = {key: value[pick] for key, value in TRAIN.items()}
            noise = torch.randn(BATCH, 10, policy.config.max_action_dim)
            if time_prior == "beta":
                time_ = 0.001 + 0.999 * torch.distributions.Beta(1.5, 1.0).sample((BATCH,))
            else:
                time_ = 0.001 + 0.998 * torch.rand(BATCH)
            optimizer_update(adapter, batch, optimizer, flow_loss_fn, noise=noise, time=time_)
        TRAINED[time_prior] = adapter.eval()
        return TRAINED[time_prior]


    def sample(adapter: SmolVLAAdapter, batch: Mapping[str, torch.Tensor], steps: int) -> tuple[torch.Tensor, float]:
        inputs = adapter.prepare_inputs(batch)
        noise = torch.randn(len(batch["action"]), 10, adapter.policy.config.max_action_dim, generator=torch.Generator().manual_seed(SEED_NOISE))
        started = time.perf_counter()
        chunk = adapter.sample_chunk(inputs, euler_sample, noise, steps)             # YOUR sampler
        return chunk, time.perf_counter() - started


    def run_trial(config: TrialConfig) -> dict[str, float | int]:
        adapter = train_model(config["time_prior"])
        with torch.no_grad():
            if config["time_prior"] not in REFERENCE_CHUNKS:
                REFERENCE_CHUNKS[config["time_prior"]] = sample(adapter, VALID, 50)[0][:, :N_ACT]
            chunk, _ = sample(adapter, VALID, config["steps"])
            executed = chunk[:, :N_ACT]
            one = {key: value[:1] for key, value in VALID.items()}
            timings = sorted(sample(adapter, one, config["steps"])[1] for _ in range(5))
        return {
            "l1_to_demo": masked_action_l1(executed, VALID["action"][:, :N_ACT], VALID["action_is_pad"][:, :N_ACT]).item(),
            "l1_to_50": (executed - REFERENCE_CHUNKS[config["time_prior"]]).abs().mean().item(),
            "ms_batch1": 1000 * timings[2],
        }


    def record(config: TrialConfig) -> None:
        config = {**BASELINE, **config}
        try:
            TRIALS.append({"config": config, "status": "ok", **run_trial(config)})
        except Exception as error:  # a failed trial is evidence too
            TRIALS.append({"config": config, "status": f"failed: {type(error).__name__}: {error}"})


    record(BASELINE)
    COLUMNS = ["l1_to_demo", "l1_to_50", "ms_batch1"]
    show_trials(TRIALS, COLUMNS);

**Your trials.** Add at least three trials. Each setting uses a fixed 300-update budget.

In [ ]:
if RUN_EXPERIMENT:
    # Reference choice of trials: one change per trial.
    MY_TRIALS = [{"steps": 1}, {"steps": 2}, {"steps": 5}, {"steps": 20}, {"steps": 50}, {"time_prior": "uniform"}]
    for trial in MY_TRIALS:
        record(trial)
    show_trials(TRIALS, COLUMNS);

In [ ]:
if RUN_EXPERIMENT:
    # Check: the protocol was followed.
    assert TRIALS[0]["config"] == BASELINE, "The first trial must be the baseline."
    assert len(TRIALS) >= 4, "Run the baseline and at least three trials of your own."
    assert all(set(t["config"]) == set(BASELINE) for t in TRIALS), "Only time_prior and steps may change."
    assert all(t["config"]["time_prior"] in {"beta", "uniform"} and t["config"]["steps"] in (1, 2, 5, 10, 20, 50) for t in TRIALS)
    ok = [t for t in TRIALS if t["status"] == "ok"]
    assert all(torch.isfinite(torch.tensor([t["l1_to_demo"], t["l1_to_50"], t["ms_batch1"]])).all() for t in ok), "Measures must be finite."
    print(f"{len(TRIALS)} trials kept ({len(ok)} ok). Budget: {STEPS} updates, batch {BATCH}, seeds {SEED_WEIGHTS}/{SEED_BATCH}/{SEED_NOISE}.")

In [ ]:
if RUN_EXPERIMENT:
    # Use your function: accuracy against time for every trial
    ok = [t for t in TRIALS if t["status"] == "ok"]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    for t in ok:
        label = f"{t['config']['time_prior'][:1]}{t['config']['steps']}"
        axes[0].scatter(t["ms_batch1"], t["l1_to_demo"]); axes[0].annotate(label, (t["ms_batch1"], t["l1_to_demo"]), fontsize=8)
        axes[1].scatter(t["ms_batch1"], t["l1_to_50"]); axes[1].annotate(label, (t["ms_batch1"], t["l1_to_50"]), fontsize=8)
    axes[0].set_xlabel("ms per chunk (batch 1)"); axes[0].set_ylabel("l1_to_demo")
    axes[1].set_xlabel("ms per chunk (batch 1)"); axes[1].set_ylabel("l1_to_50")
    plt.tight_layout(); plt.show()

**Explain (2–4 sentences, edit this cell).** Which time prior and step count would you pick, and what is the cost? Flow matching has no noise schedule and no random step noise: what does that change about repeating a sample? What can you *not* conclude from a tiny model with random VLM weights?

*Optional extension:* repeat the baseline and one other trial with another `SEED_NOISE`; keep all rows.

## Opt-in A: your flow loss on the prepared learner at real size

This loads the **prepared learner** (450M parameters, `smolvla-c2-5000-development`) from local files, builds one real batch with the real pipeline (two 256×256 frames, state, the instruction, 50 actions), and compares **your** `flow_loss_fn` with the pinned `policy(batch, noise, time)` for the same noise and time. It then makes `REAL_UPDATES` AdamW update(s) with your loss on the expert only (the VLM must not move) and prints the loss before and after. One or two updates are an integration check, not training evidence; nothing is saved. The code selects CUDA when available and otherwise uses CPU. It loads one model. Set `RUN_REAL_FLOW_LOSS = True` to run it.

In [ ]:
RUN_REAL_FLOW_LOSS = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REAL_UPDATES = 1

if RUN_REAL_FLOW_LOSS:
    from torch.utils.data import default_collate

    learner, preprocessor, _ = load_prepared_smolvla("learner", DATA, device=DEVICE)
    dataset = open_libero_dataset(CONFIGS, DATA, "bowl_plate", "train", {"action": [i / 10 for i in range(learner.config.chunk_size)]}, episodes=[379])
    batch = preprocessor(default_collate([dataset[7]]))
    adapter = SmolVLAAdapter(learner)
    noise = torch.randn(1, learner.config.chunk_size, learner.config.max_action_dim, generator=torch.Generator().manual_seed(0)).to(DEVICE)
    time_ = torch.tensor([0.6], device=DEVICE)
    with torch.no_grad():
        pinned, _ = learner(dict(batch), noise=noise, time=time_)
        yours, _ = flow_loss_fn(adapter, dict(batch), noise=noise, time=time_)
    print("pinned loss:", float(pinned), "| your loss:", float(yours))
    assert abs(float(pinned) - float(yours)) < 1e-4, "Your loss must equal the pinned loss for the same noise and time."
    for parameter in learner.parameters():
        parameter.requires_grad_(True)
    params = select_trainable_parameters(learner)
    adapter = SmolVLAAdapter(learner, train_mode_fn=train_mode_with_frozen_vlm)
    optimizer = torch.optim.AdamW(params, lr=1e-5)
    vlm_probe = next(p for n, p in learner.named_parameters() if n.startswith("model.vlm_with_expert.vlm."))
    head_probe = learner.model.action_out_proj.weight
    before = (vlm_probe.detach().clone(), head_probe.detach().clone())
    history = [optimizer_update(adapter, dict(batch), optimizer, flow_loss_fn, noise=noise, time=time_)["loss"] for _ in range(REAL_UPDATES)]
    print("loss before the updates:", float(pinned), "| losses during the update(s):", [round(h, 5) for h in history])
    print("VLM weight unchanged:", torch.equal(before[0], vlm_probe), "| action head changed:", not torch.equal(before[1], head_probe))
else:
    print("Real flow-loss check is off. Set RUN_REAL_FLOW_LOSS=True to run it (loads one 450M model).")

## Opt-in B: your sampler in closed loop on development states 10–14

This runs a prepared policy with **your** `euler_sample` (through `CallbackSmolVLAPolicy`, which uses the pinned vision-language prefix and action expert but not the pinned sampler) in the LIBERO simulator on the *methods* development states `10–14` only (states `0–9` are for exploration, `15–19` for the intervention unit; the final states `20–49` are never used here). The reset seed is `20260930 + state`, the policy noise seed is the same, and every attempt, including failures and errors, is kept in `ATTEMPTS`. The instruction is the task's own text. The controller runs at 20 Hz and the demonstrations were recorded at 10 Hz; the policy is **not** resampled. This uses one 450M model and a bounded 300-step episode. Set `RUN_DEV_STATE_ROLLOUTS = True` to run it. Saved results of the same policies, produced earlier by the project, are printed for orientation and are **not** produced by this notebook.

In [ ]:
RUN_DEV_STATE_ROLLOUTS = False
ROLLOUT_POLICY = "learner"             # "learner" (trained in this project) or "published" (smolvla-libero)
DEV_STATES = (10, 11, 12, 13, 14)      # narrow this list to run fewer episodes
ROLLOUT_STEPS = (10, 3)                # Euler steps to compare (10 is the pinned default)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_STEPS = 300
RESET_BASE_SEED = 20260930
ATTEMPTS = []

if RUN_DEV_STATE_ROLLOUTS:
    import pandas as pd
    from vla_lab.sim.episode import open_libero_task, run_policy_episode

    assert all(10 <= s <= 14 for s in DEV_STATES), "Use development states 10-14 only in this unit."
    task_text = yaml.safe_load((CONFIGS / "data" / "episodes.yaml").read_text())["tasks"]["bowl_plate"]["task_name"]
    policy, preprocessor, postprocessor = load_prepared_smolvla(ROLLOUT_POLICY, DATA, device=DEVICE, n_action_steps=10)
    for steps in ROLLOUT_STEPS:
        for state_id in DEV_STATES:
            wrapper = CallbackSmolVLAPolicy(policy, euler_sample, num_steps=steps, seed=RESET_BASE_SEED + state_id)
            environment = None
            try:
                environment = open_libero_task("bowl_plate", state_id, CONFIGS, DATA)
                result = run_policy_episode(environment, wrapper, preprocessor, postprocessor, "bowl_plate", state_id, CONFIGS,
                                            partition="development", max_steps=MAX_STEPS, task_prompt=task_text, reset_seed=RESET_BASE_SEED + state_id)
                ATTEMPTS.append({"steps": steps, "state": state_id, "success": result.success, "env_steps": result.steps, "ended": result.termination_reason,
                                 "chunks": wrapper.chunks_made, "mean_sampler_ms": 1000 * sum(wrapper.sample_seconds) / max(len(wrapper.sample_seconds), 1)})
            except Exception as error:  # keep failed attempts
                ATTEMPTS.append({"steps": steps, "state": state_id, "success": False, "ended": f"error: {type(error).__name__}: {error}"})
            finally:
                if environment is not None:
                    try:
                        environment.close()
                    except Exception as error:
                        ATTEMPTS.append({"steps": steps, "state": state_id, "success": False,
                                         "ended": f"close error: {type(error).__name__}: {error}"})
    table = pd.DataFrame(ATTEMPTS)
    print(table.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
    print(table.groupby("steps")["success"].agg(["sum", "count"]).rename(columns={"sum": "successes", "count": "attempts"}))
    saved = yaml.safe_load((WORKSPACE / "manifests/smolvla_reference.yaml").read_text())["published_reference"]["selected_result"]
    print("Saved published-policy result (development states 0-19, 10 executed actions; NOT produced here):", saved["successes"], "of", saved["episodes"])
else:
    print("Closed-loop rollouts are off. Set RUN_DEV_STATE_ROLLOUTS=True to run", len(DEV_STATES) * len(ROLLOUT_STEPS), "episodes (one 450M model).")

## Opt-in C: adapt the prepared learner and test its sampler

This compares two learning rates from the same prepared learner weights. Each setting uses the same two training examples, batch size 1, five updates, noise, and time values. The Unit 12B `flow_loss_fn` updates the selected action parameters through `optimizer_update`; the VLM stays frozen. The code measures held-out loss before and after training. It then runs your Euler sampler for at most 40 simulator steps on development state 10. Every training and simulator failure stays in `REAL_ADAPTATION_RESULTS`. The code loads one model at a time and writes no weights. It uses local files only. Set `RUN_REAL_ADAPTATION = True` to run these two bounded trials.

In [ ]:
RUN_REAL_ADAPTATION = False
ADAPTATION_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ADAPTATION_LRS = (5e-6, 1e-5)
ADAPTATION_STEPS = 5
ADAPTATION_SEED = 19
REAL_ADAPTATION_RESULTS = []

if RUN_REAL_ADAPTATION:
    import hashlib
    from torch.utils.data import default_collate
    from vla_lab.teaching.act_model import open_libero_dataset
    from vla_lab.sim.episode import open_libero_task, run_policy_episode

    train_indices, validation_indices = (3, 40), (7, 12)
    timestamps = {"action": [i / 10 for i in range(50)]}
    setup_error = None
    try:
        train_data = open_libero_dataset(CONFIGS, DATA, "bowl_plate", "train", timestamps, episodes=[379])
        validation_data = open_libero_dataset(CONFIGS, DATA, "bowl_plate", "validation", timestamps)
    except Exception as error:
        train_data = validation_data = None
        setup_error = f"dataset setup failed: {type(error).__name__}: {error}"
    checkpoint_file = DATA / "runs/smolvla-c2-5000-development/checkpoint-005000/model.safetensors"
    checkpoint_hash = None
    try:
        if not checkpoint_file.is_file():
            raise FileNotFoundError(f"Prepared SmolVLA checkpoint missing: {checkpoint_file}")
        file_digest = hashlib.sha256()
        with checkpoint_file.open("rb") as checkpoint_stream:
            for block in iter(lambda: checkpoint_stream.read(1024 * 1024), b""):
                file_digest.update(block)
        checkpoint_hash = file_digest.hexdigest()
    except Exception as error:
        setup_error = setup_error or f"checkpoint setup failed: {type(error).__name__}: {error}"

    def module_hash(module: torch.nn.Module) -> str:
        digest = hashlib.sha256()
        for name, parameter in module.named_parameters():
            digest.update(name.encode())
            value = parameter.detach().to(device="cpu", copy=True).contiguous()
            digest.update(value.view(torch.uint8).numpy().tobytes())
        return digest.hexdigest()

    previous_initial_head = None
    for trial_index, learning_rate in enumerate(ADAPTATION_LRS):
        result = {"lr": learning_rate, "updates": ADAPTATION_STEPS, "checkpoint_sha256": checkpoint_hash,
                  "status": "failed", "rollout": {"state": 10, "max_steps": 40, "status": "not run"}}
        policy = adapter = environment = None
        parameters = optimizer = preprocessor = postprocessor = training_batches = validation_batch = None
        train_noises = train_times = validation_noise = validation_time = None
        noise_generator = head_before = initial_head = frozen_vlm = wrapper = batch = None
        before_loss = after_loss = vlm_hash_before = vlm_hash_after = episode = task_text = rollout = None
        try:
            if setup_error:
                raise RuntimeError(setup_error)
            policy, preprocessor, postprocessor = load_prepared_smolvla("learner", DATA, device=ADAPTATION_DEVICE, n_action_steps=10)
            torch.manual_seed(ADAPTATION_SEED)
            initial_head = policy.model.action_out_proj.weight.detach().clone()
            if previous_initial_head is None:
                previous_initial_head = initial_head.detach().cpu().clone()
            else:
                assert torch.equal(initial_head.cpu(), previous_initial_head), "Every learning-rate trial must start from the same learner weights."
            for parameter in policy.parameters():
                parameter.requires_grad_(True)
            parameters = select_trainable_parameters(policy)
            adapter = SmolVLAAdapter(policy, train_mode_fn=train_mode_with_frozen_vlm)
            training_batches = [preprocessor(default_collate([train_data[index]])) for index in train_indices]
            validation_batch = preprocessor(default_collate([validation_data[index] for index in validation_indices]))
            optimizer = torch.optim.AdamW(parameters, lr=learning_rate)
            noise_generator = torch.Generator(device=ADAPTATION_DEVICE).manual_seed(ADAPTATION_SEED)
            train_noises = [torch.randn(1, policy.config.chunk_size, policy.config.max_action_dim, device=ADAPTATION_DEVICE,
                                        generator=noise_generator) for _ in range(ADAPTATION_STEPS)]
            train_times = [torch.tensor([value], device=ADAPTATION_DEVICE)
                           for value in (0.25, 0.5, 0.75, 0.4, 0.6)]
            validation_noise = torch.randn(len(validation_indices), policy.config.chunk_size, policy.config.max_action_dim,
                                           device=ADAPTATION_DEVICE, generator=noise_generator)
            validation_time = torch.full((len(validation_indices),), 0.5, device=ADAPTATION_DEVICE)
            adapter.eval()
            frozen_vlm = policy.model.vlm_with_expert.vlm
            vlm_hash_before = module_hash(frozen_vlm)
            with torch.no_grad():
                before_loss, _ = flow_loss_fn(adapter, validation_batch, noise=validation_noise, time=validation_time)
            head_before = policy.model.action_out_proj.weight.detach().clone()
            for step in range(ADAPTATION_STEPS):
                batch = training_batches[step % len(training_batches)]
                optimizer_update(adapter, batch, optimizer, flow_loss_fn, noise=train_noises[step], time=train_times[step])
            adapter.eval()
            with torch.no_grad():
                after_loss, _ = flow_loss_fn(adapter, validation_batch, noise=validation_noise, time=validation_time)
            vlm_frozen = all(not parameter.requires_grad and parameter.grad is None for parameter in frozen_vlm.parameters())
            vlm_hash_after = module_hash(frozen_vlm)
            expert_changed = not torch.equal(head_before, policy.model.action_out_proj.weight)
            result.update({"status": "trained", "val_loss_before": float(before_loss), "val_loss_after": float(after_loss),
                           "expert_changed": expert_changed, "vlm_frozen": vlm_frozen,
                           "vlm_hash_unchanged": vlm_hash_before == vlm_hash_after})
            assert expert_changed, "The learner update did not change the action head."
            assert vlm_frozen, "A VLM parameter is trainable or has a gradient."
            assert vlm_hash_before == vlm_hash_after, "The frozen VLM weights changed."

            rollout = {"lr": learning_rate, "state": 10, "max_steps": 40, "status": "failed"}
            try:
                task_text = yaml.safe_load((CONFIGS / "data" / "episodes.yaml").read_text())["tasks"]["bowl_plate"]["task_name"]
                environment = open_libero_task("bowl_plate", 10, CONFIGS, DATA)
                wrapper = CallbackSmolVLAPolicy(policy, euler_sample, num_steps=10, seed=20260930 + 10)
                episode = run_policy_episode(environment, wrapper, preprocessor, postprocessor, "bowl_plate", 10, CONFIGS,
                                             partition="development", max_steps=40, task_prompt=task_text, reset_seed=20260930 + 10)
                rollout.update({"status": "ok", "success": episode.success, "steps": episode.steps,
                                "ended": episode.termination_reason, "chunks": wrapper.chunks_made})
            except Exception as error:
                rollout["error"] = f"{type(error).__name__}: {error}"
            finally:
                if environment is not None:
                    try:
                        environment.close()
                    except Exception as error:
                        rollout["close_error"] = f"{type(error).__name__}: {error}"
            result["rollout"] = rollout
        except Exception as error:
            result["status"] = f"failed: {type(error).__name__}: {error}"
        finally:
            REAL_ADAPTATION_RESULTS.append(result)
            del parameters, optimizer, preprocessor, postprocessor, training_batches, validation_batch
            del train_noises, train_times, validation_noise, validation_time
            del noise_generator, head_before, initial_head, frozen_vlm, wrapper, batch
            del before_loss, after_loss, vlm_hash_before, vlm_hash_after, episode, task_text, rollout, environment
            if policy is not None:
                del policy
            if adapter is not None:
                del adapter
            if ADAPTATION_DEVICE == "cuda":
                torch.cuda.empty_cache()
    print(json.dumps(REAL_ADAPTATION_RESULTS, indent=2))
else:
    print("Real adaptation is off. Set RUN_REAL_ADAPTATION=True to run two five-update trials and state-10 rollouts.")

## What you built

- `flow_training_pair`, `masked_velocity_loss`, `flow_loss_fn` and `euler_sample`: the flow-matching objective and sampler of SmolVLA, equal to the pinned loss and integrator, running through the real prefix cache and action expert, with the VLM frozen.
- A policy object whose actions come from your sampler, with fixed noise, step budgets, timing and every trial or attempt kept.

**Next:** the comparison and runtime units reuse these functions: package versions of everything in Units 7–12 are available from `vla_lab.teaching`.